In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ============================================
# SAFE EXPLAINABILITY-GUIDED COLON CANCER GUI
# ============================================

import numpy as np
import tensorflow as tf
import cv2
import gradio as gr
from tensorflow.keras.applications.efficientnet import preprocess_input
from google.colab import drive

# ============================================
# 1️⃣ MOUNT DRIVE & LOAD MODEL
# ============================================

drive.mount('/content/drive')

MODEL_PATH = "/content/drive/MyDrive/Explainability_Guided_Colon_Cancer_Detection_EfficientNetB0_Final.h5"
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

print("✅ Model loaded successfully")

# ============================================
# 2️⃣ DOMAIN VALIDATION (COLON IMAGE CHECK)
# ============================================

def is_colon_like_image(image):
    """
    Reject non-histopathology images (skull, cartoons, photos, etc.)
    Based on color & texture characteristics of H&E stained slides.
    """
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    hue = hsv[:, :, 0]
    saturation = hsv[:, :, 1]

    sat_mean = np.mean(saturation)
    hue_mean = np.mean(hue)

    # Reject grayscale / low-texture images
    if sat_mean < 20:
        return False

    # Accept pink-purple H&E color ranges
    if not (90 < hue_mean < 180):
        return False

    return True

# ============================================
# 3️⃣ GRAD-CAM FUNCTION
# ============================================

def grad_cam(model, img_array, conv_layer_name="block7a_project_conv"):
    conv_layer = model.get_layer(conv_layer_name)

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[conv_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.nn.relu(heatmap)
    heatmap /= (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()

# ============================================
# 4️⃣ PREDICTION FUNCTION (SAFE LOGIC)
# ============================================

def predict_with_gradcam(image):

    # 🚫 Reject non-colon images
    if not is_colon_like_image(image):
        return (
            "<div style='font-size:24px; font-weight:bold; color:red;'>"
            "⚠️ Invalid Input Image</div>"
            "<div style='font-size:16px;'>"
            "This image is NOT a colon histopathology image.</div><br>"
            "<div style='font-size:14px; color:gray;'>"
            "Please upload a valid H&E-stained colon tissue image.</div>",
            image
        )

    # ✅ Preprocess
    img = cv2.resize(image, (224, 224))
    img_array = np.expand_dims(img, axis=0)
    img_array = preprocess_input(img_array)

    # ✅ Prediction
    score = float(model.predict(img_array, verbose=0)[0][0])
    probability = score * 100

    # 🔐 Safety decision logic
    if probability >= 60:
        label = "🔴 Cancer (Adenocarcinoma)"
        color = "red"
    elif probability <= 40:
        label = "🟢 Normal Colon Tissue"
        color = "green"
    else:
        label = "⚠️ Uncertain Prediction"
        color = "orange"

    # Confidence interpretation
    if probability >= 90:
        confidence = "Very High"
    elif probability >= 70:
        confidence = "High"
    elif probability >= 60 or probability <= 40:
        confidence = "Moderate"
    else:
        confidence = "Low / Uncertain"

    # 🔥 Grad-CAM
    heatmap = grad_cam(model, img_array)
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_color = cv2.applyColorMap(
        np.uint8(255 * heatmap_resized),
        cv2.COLORMAP_JET
    )

    overlay = cv2.addWeighted(
        cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
        0.6,
        heatmap_color,
        0.4,
        0
    )
    overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

    # 🧾 Output
    result_html = f"""
    <div style="font-size:24px; font-weight:bold; color:{color};">
        Prediction: {label}
    </div>
    <div style="font-size:18px;">
        Probability: <b>{probability:.2f}%</b>
    </div>
    <div style="font-size:18px;">
        Confidence Level: <b>{confidence}</b>
    </div>
    <br>
    <div style="font-size:14px; color:gray;">
        ⚠️ This AI system is trained ONLY on colon histopathology images.<br>
        Predictions on non-colon images are intentionally rejected.<br>
        For research and decision-support use only.
    </div>
    """

    return result_html, overlay

# ============================================
# 5️⃣ GRADIO INTERFACE
# ============================================

interface = gr.Interface(
    fn=predict_with_gradcam,
    inputs=gr.Image(type="numpy", label="Upload Colon Histopathology Image"),
    outputs=[
        gr.HTML(label="Prediction Result"),
        gr.Image(label="Grad-CAM Visualization")
    ],
    title="Explainability-Guided Deep Learning Framework for Colon Cancer Detection Using Histopathological Images ",
    description=(
        "<b>Upload a colon histopathology image (H&E stained).</b><br><br>"
        "🔴 ≥ 60% → Cancer<br>"
        "🟢 ≤ 40% → Normal<br>"
        "⚠️ 40–60% → Uncertain<br><br>"
        "Non-colon images are automatically rejected."
    ),
    theme="soft"
)

interface.launch(share=True)
